In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 17:40:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [3]:
# Membuat dataset yang lebih besar (20.000 baris) khusus untuk latihan ini
np.random.seed(11)
n = 20000
data_besar = {
    "order_id": [f"O{i}" for i in range(n)],
    "customer_id": np.random.randint(1, 21, size=n),
    "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"], size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"], size=n),
    "metode_pembayaran": np.random.choice(["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
pd.DataFrame(data_besar).to_csv("data_transaksi_besar.csv", index=False)

csv_size = os.path.getsize("data_transaksi_besar.csv")
print(f"Ukuran CSV: {csv_size:,} bytes ({csv_size/1024:.1f} KB) untuk {n:,} baris")

Ukuran CSV: 892,909 bytes (872.0 KB) untuk 20,000 baris


In [3]:
# Membaca CSV, lalu menuliskannya ulang sebagai Parquet
df_besar = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_besar.write.mode("overwrite").parquet("data_transaksi_besar_parquet")

# Menghitung total ukuran seluruh berkas .parquet yang dihasilkan
parquet_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk("data_transaksi_besar_parquet")
    for f in fn if f.endswith(".parquet")
)

print(f"Ukuran CSV      : {csv_size:,} bytes ({csv_size/1024:.1f} KB)")
print(f"Ukuran Parquet  : {parquet_size:,} bytes ({parquet_size/1024:.1f} KB)")
print(f"Parquet {(1 - parquet_size/csv_size)*100:.1f}% lebih kecil dibanding CSV")

[Stage 2:>                                                          (0 + 1) / 1]

Ukuran CSV      : 892,909 bytes (872.0 KB)
Ukuran Parquet  : 202,197 bytes (197.5 KB)
Parquet 77.4% lebih kecil dibanding CSV


In [4]:
# Membaca kembali Parquet — perhatikan kita TIDAK perlu inferSchema=True
# karena skema/tipe data sudah tersimpan built-in di dalam berkas Parquet itu sendiri
df_dari_parquet = spark.read.parquet("data_transaksi_besar_parquet")
df_dari_parquet.printSchema()
df_dari_parquet.show(5)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)

+--------+-----------+----------+---------+-----------------+----------+
|order_id|customer_id|  kategori|     kota|metode_pembayaran|pendapatan|
+--------+-----------+----------+---------+-----------------+----------+
|      O0|         17|   Makanan|     Solo|              COD|    196650|
|      O1|         18| Kesehatan| Magelang|         E-Wallet|    283512|
|      O2|         14|Elektronik| Semarang|              COD|    108835|
|      O3|         13| Kesehatan|Purworejo|     Kartu Kredit|    390604|
|      O4|          2|   Fashion|Purworejo|         E-Wallet|    427224|
+--------+-----------+----------+---------+-----------------+----------+
only showing top 5 rows



In [5]:
# Menulis ulang dengan partisi berdasarkan kolom 'kota'
df_besar.write.mode("overwrite").partitionBy("kota").parquet("data_transaksi_partisi_kota")

# Melihat struktur folder yang dihasilkan
!ls data_transaksi_partisi_kota

'kota=Magelang'   'kota=Semarang'  'kota=Yogyakarta'
'kota=Purworejo'  'kota=Solo'	    _SUCCESS


In [6]:
import json

# Membuat data pelanggan sebagai JSON Lines (satu objek JSON per baris — format umum untuk big data)
np.random.seed(21)
pelanggan = [
    {
        "customer_id": i,
        "nama": f"Pelanggan{i}",
        "membership": np.random.choice(["Silver", "Gold", "Platinum"]),
        "kota_domisili": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"])
    }
    for i in range(1, 21)
]

with open("data_pelanggan.json", "w") as f:
    for p in pelanggan:
        f.write(json.dumps(p) + "\n")

print("Berkas data_pelanggan.json berhasil dibuat (format JSON Lines).")

Berkas data_pelanggan.json berhasil dibuat (format JSON Lines).


In [7]:
# Membaca JSON menggunakan Spark — Spark otomatis mendeteksi struktur/skema JSON
df_pelanggan = spark.read.json("data_pelanggan.json")
df_pelanggan.printSchema()
df_pelanggan.show(5)

root
 |-- customer_id: long (nullable = true)
 |-- kota_domisili: string (nullable = true)
 |-- membership: string (nullable = true)
 |-- nama: string (nullable = true)

+-----------+-------------+----------+----------+
|customer_id|kota_domisili|membership|      nama|
+-----------+-------------+----------+----------+
|          1|     Magelang|      Gold|Pelanggan1|
|          2|     Magelang|    Silver|Pelanggan2|
|          3|     Magelang|    Silver|Pelanggan3|
|          4|         Solo|    Silver|Pelanggan4|
|          5|         Solo|      Gold|Pelanggan5|
+-----------+-------------+----------+----------+
only showing top 5 rows



In [8]:
# EXTRACT — membaca dari berbagai sumber
df_transaksi_etl = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_pelanggan_etl = spark.read.json("data_pelanggan.json")

print("Data transaksi:", df_transaksi_etl.count(), "baris")
print("Data pelanggan:", df_pelanggan_etl.count(), "baris")

Data transaksi: 20000 baris
Data pelanggan: 20 baris


In [9]:
from pyspark.sql.functions import when

# TRANSFORM — menggabungkan, membersihkan, dan memperkaya data
df_etl = df_transaksi_etl.join(df_pelanggan_etl, on="customer_id", how="left")

# Menambahkan kolom turunan: kategori nilai transaksi
df_etl = df_etl.withColumn(
    "segmen_transaksi",
    when(col("pendapatan") >= 300000, "Tinggi")
    .when(col("pendapatan") >= 150000, "Sedang")
    .otherwise("Rendah")
)

# Memastikan tidak ada data pelanggan yang gagal ter-join (data quality check sederhana)
jumlah_tanpa_pelanggan = df_etl.filter(col("nama").isNull()).count()
print(f"Transaksi tanpa data pelanggan yang cocok: {jumlah_tanpa_pelanggan}")

df_etl.select("order_id", "nama", "membership", "kategori", "pendapatan", "segmen_transaksi").show(5)

Transaksi tanpa data pelanggan yang cocok: 0
+--------+-----------+----------+----------+----------+----------------+
|order_id|       nama|membership|  kategori|pendapatan|segmen_transaksi|
+--------+-----------+----------+----------+----------+----------------+
|      O0|Pelanggan17|      Gold|   Makanan|    196650|          Sedang|
|      O1|Pelanggan18|  Platinum| Kesehatan|    283512|          Sedang|
|      O2|Pelanggan14|  Platinum|Elektronik|    108835|          Rendah|
|      O3|Pelanggan13|      Gold| Kesehatan|    390604|          Tinggi|
|      O4| Pelanggan2|    Silver|   Fashion|    427224|          Tinggi|
+--------+-----------+----------+----------+----------+----------------+
only showing top 5 rows



In [10]:
# LOAD — menyimpan hasil akhir ke HDFS dalam format Parquet, terpartisi berdasarkan kategori
!hdfs dfs -mkdir -p /user/mahasiswa/etl_output

df_etl.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/mahasiswa/etl_output/transaksi_enriched"
)

print("Pipeline ETL selesai — hasil tersimpan di HDFS.")
!hdfs dfs -ls /user/mahasiswa/etl_output/transaksi_enriched

Pipeline ETL selesai — hasil tersimpan di HDFS.
Found 6 items
-rw-r--r--   3 azka supergroup          0 2026-09-24 13:14 /user/mahasiswa/etl_output/transaksi_enriched/_SUCCESS
drwxr-xr-x   - azka supergroup          0 2026-09-24 13:14 /user/mahasiswa/etl_output/transaksi_enriched/kategori=Elektronik
drwxr-xr-x   - azka supergroup          0 2026-09-24 13:14 /user/mahasiswa/etl_output/transaksi_enriched/kategori=Fashion
drwxr-xr-x   - azka supergroup          0 2026-09-24 13:14 /user/mahasiswa/etl_output/transaksi_enriched/kategori=Kesehatan
drwxr-xr-x   - azka supergroup          0 2026-09-24 13:14 /user/mahasiswa/etl_output/transaksi_enriched/kategori=Makanan
drwxr-xr-x   - azka supergroup          0 2026-09-24 13:14 /user/mahasiswa/etl_output/transaksi_enriched/kategori=Rumah Tangga


In [4]:
df_final = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/etl_output/transaksi_enriched")
print("Jumlah baris hasil akhir:", df_final.count())

# Ringkasan cepat sebagai bukti data siap dipakai untuk analisis lebih lanjut
df_final.groupBy("segmen_transaksi").agg(
    count("order_id").alias("jumlah_transaksi"),
    avg("pendapatan").alias("rata_rata_pendapatan")
).orderBy("segmen_transaksi").show()

Jumlah baris hasil akhir: 20000


+----------------+----------------+--------------------+
|segmen_transaksi|jumlah_transaksi|rata_rata_pendapatan|
+----------------+----------------+--------------------+
|          Rendah|            4464|   99639.28001792115|
|          Sedang|            6633|  224445.05759083372|
|          Tinggi|            8903|   398577.4426597776|
+----------------+----------------+--------------------+

